## Nikola Stojanovic

## Assignment 3: Spark

## Learning Outcomes

In this assignment, you will do the following:

* Import a dataset into the Databricks Spark environment

* Create tables for the data imported

* Perform basic data analysis using transformations and Spark SQL

**Setup Instructions**

1. Read the article:
> https://databricks.com/blog/2016/07/14/a-tale-of-three-apache-spark-apis-rdds-dataframes-and-datasets.html (links to an external site)


2. Download Data: You an either download the JSON file from the Assignment 3 home page in Learn under 'EXPECTATIONS AND ASSIGNMENT FILES' or you can download the file from Github to your computer: 
> https://github.com/dmatrix/examples/blob/master/spark/databricks/notebooks/py/data/iot_devices.json (links to an external site) Click on the raw data source -> right click -> click save as. The file will download locally and now you can import it to Databricks. **NOTE**: 

3. Import files: How to import your downloaded files (from step 3) to your Databricks cluster: https://www.projectpro.io/recipes/create-dataframe-from-json-file-read-data-from-dbfs-and-write-into-dbfs (links to an external site).


4. Import the notebook below.  There is some data exploration already done in this notebook for your reference. For details on how to import the notebook above see: https://docs.databricks.com/user-guide/notebooks/index.html (links to an external site).
> https://databricks-prod-cloudfront.cloud.databricks.com/public/4027ec902e239c93eaaa8714f173bcfc/5915990090493625/411609171004360/6085673883631125/latest.html


5. Run it. **NOTE**: Don't forget to create a cluster and attach the imported notebook to it (left upper corner: button `detached`) before trying to run it.

**Questions** (12 marks)

1. Explain the main differences between RDDs, Dataframes and Datasets (4 marks)


2. Answer the following questions:

   2.1 How many sensor pads are reported to be from Poland (2 marks) 
   
   2.2 How many different LCDs (distinct colors) are present in the dataset (2 marks)  
   
   2.3 Find 5 countries that have the largest number of MAC devices used (2 marks)
   
   2.4 Propose and try an interesting statistical test or machine learning model you could use to gain insight from this dataset. Note, you don't have to use Machine Learning for this question. You can apply any analysis to the data even using SparkSQL, Python visualization libraries to analyze the data. Another example cloud be to apply correlation functions or other Spark functions to analyze the data. (2 marks)
   

**NOTE**: You may use MLLib in 2.4: https://spark.apache.org/docs/latest/ml-guide.html. Marks are awarded for the idea and implementation of the test/ML model.  

Please submit the **published** notebook link in a word/pdf document. Do not submit HTML, Jupyter notebook, or archive (DBC) formats.  


In [0]:
#1.  Explain the main differences between RDDs, Dataframes and Datasets (4 marks)




In [0]:
#imports
from pyspark.sql.functions import col, asc, count


# read the json file and create the dataframe
 
 
file_location = "/Volumes/workspace/default/assignment3"
file_type = "json"
 
# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","
 
# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)
 
display(df)

In [0]:
# Create a view or table
 
temp_table_name = "iot_devices_json"
 
df.createOrReplaceTempView(temp_table_name)
print(temp_table_name)

In [0]:
df.write.mode("overwrite").saveAsTable("iot_devices_table")
print("Table 'iot_devices_table' created")

In [0]:
%sql
SELECT COUNT(device_name) AS Num_of_Devices
FROM workspace. default.iot_devices_table
WHERE cn = 'Poland' AND device_name LIKE "sensor-pad%";

In [0]:
%sql
SELECT distinct (lcd)
FROM workspace.default.iot_devices_table


In [0]:
df.count()
lcd_colours = df.filter(col("lcd").isNotNull()).select("lcd").distinct()
display(lcd_colours)

In [0]:
display(df)

### ## Data exploration

In [0]:
df.take(5)

In [0]:

#2.1 How many sensor pads are reported to be from Poland (2 marks)
sensorPad_Poland_count = df.filter(col("cn") == "Poland").filter(col("device_name").like("%sensor-pad%")).count()
display(sensorPad_Poland_count)




In [0]:
#2.2 How many different LCDs (distinct colors) are present in the dataset (2 marks)

lcd_colours = df.filter(col("lcd").isNotNull()).select("lcd").distinct()
display(lcd_colours)


### 2.3 Find 5 countries that have the largest number of MAC devices used (2 marks)

In [0]:

countries_MACdevice = df.select("cn","device_name").filter(col("device_name").like("device-mac%")).groupBy("cn").agg(count("device_name").alias("MACDevices_Used")).orderBy("MACDevices_Used", ascending=False).take(5)

display(countries_MACdevice)


#df.groupBy("key").agg(count("value").alias("count")).orderBy("key").collect()

In [0]:
%sql
SELECT cn AS Country, COUNT(device_name) AS MAC_Devices_Used
FROM workspace.default.iot_devices_table
WHERE device_name LIKE "device-mac%"
GROUP BY cn
ORDER BY COUNT(device_name) DESC
LIMIT 5

### 2.4 Propose and try an interesting statistical test or machine learning model you could use to gain insight from this dataset. Note, you don't have to use Machine Learning for this question. You can apply any analysis to the data even using SparkSQL, Python visualization libraries to analyze the data. Another example cloud be to apply correlation functions or other Spark functions to analyze the data. (2 marks)

In [0]:
#display(df.select("cn", "device_name").take(10))
#3 What is the maximum temperature reported in the dataset (2 marks)
#df.select("temp").agg({"temp": "max"}).show()
#2
countries_device = df.select("cn","device_name").filter(col("device_name").like("device-mac%"))
display(countries_device)

In [0]:
# issue select, map, filter operations on the dataframes
 

TempFilter = df.filter(col("temp") > 30).filter(col("humidity") > 70)
display(TempFilter)

In [0]:
TempFilter10 = df.filter(col("temp") > 30).filter(col("humidity") > 70).take(10)
TempFilter10

In [0]:
# Mapping four fields- temp, device_name, device_id, cca3
dfTempMap = df.where((col("temp") > 25)).rdd.map(lambda d: (d.temp, d.device_name, d.device_id, d.cca3))
dfTempMap